In [1]:
import urllib
from pathlib import Path

In [2]:
# 1. Local paths
NEPAL_GEO_DIR = Path().resolve().parent / "data" / "Country" 
NEPAL_GEO_DIR.mkdir(parents=True, exist_ok=True)
NEPAL_GEO_PATH = NEPAL_GEO_DIR / "nepal.geojson"

In [3]:
url = "https://datahub.io/core/geo-boundaries-world-110m/_r/-/countries.geojson"

In [21]:
_WEMBLEY_COORDINATES = {"LAT": 51.548970, "LON": -0.279222}
_BHARATPUR_COORDINATES = {"LAT": 27.676546, "LON": 84.463889}
_TIBET_COORDINATES = {"LAT": 9.334221, "LON": 83.576237}
_GANDAKI_PROVINCE = {"LAT": 29.253410, "LON": 83.923084}

In [15]:
import geopandas as gpd
from shapely.geometry import Point

if not NEPAL_GEO_PATH.exists():
    print("Downloading global boundaries from DataHub...")
    
    # Download world file temporarily
    world_geojson_path = NEPAL_GEO_DIR / "world.geojson"
   # Use User-Agent header to prevent GitHub raw blocking
    req = urllib.request.Request(url, headers={'User-Agent': 'Mozilla/5.0'})
    with urllib.request.urlopen(req) as response, open(world_geojson_path, 'wb') as out_file:
        out_file.write(response.read())
    
    print("Extracting Nepal from world boundaries...")
    world_gdf = gpd.read_file(world_geojson_path)
    
    # Filter for Nepal (DataHub uses 'name' or ISO codes depending on version; let's check both)
    nepal_gdf = world_gdf[world_gdf['name'].str.lower() == 'nepal']
    
    if nepal_gdf.empty:
        # Fallback filter by ISO code if available
        nepal_gdf = world_gdf[world_gdf.get('ISO_A3', world_gdf.get('iso_a3', '')) == 'NPL']
        
    # Save just Nepal's geometry locally to reuse forever
    nepal_gdf.to_file(NEPAL_GEO_PATH, driver="GeoJSON")
    
    # Clean up the large world file
    world_geojson_path.unlink(missing_ok=True)
    print(f"Nepal boundary successfully saved to: {NEPAL_GEO_PATH}")

# 4. Load the cached Nepal boundary into memory for your API
print("Loading Nepal boundary into memory...")
_nepal_gdf = gpd.read_file(NEPAL_GEO_PATH)
_nepal_boundary = _nepal_gdf.unary_union
print("Ready for point-in-polygon checks!")

def is_inside_nepal(lat: float, lon: float) -> bool:
    """Checks if coordinates fall within Nepal using the cached boundary."""
    point = Point(lon, lat)  # Remember: Longitude first, Latitude second
    return _nepal_boundary.contains(point) or _nepal_boundary.touches(point)

Loading Nepal boundary into memory...
Ready for point-in-polygon checks!


/var/folders/bl/c9xq73wj61b5k_gjtrh9b0zh0000gn/T/ipykernel_18230/2543347562.py:34: DeprecationWarning: The 'unary_union' attribute is deprecated, use the 'union_all()' method instead.
  _nepal_boundary = _nepal_gdf.unary_union


In [22]:
is_inside_nepal(_GANDAKI_PROVINCE["LAT"], _GANDAKI_PROVINCE["LON"])

True